In [ ]:
from google.colab import drive
drive.mount('/content/drive') #

Mounted at /content/drive


In [ ]:
!git clone https://github.com/Goshuatechn/2026-02-IC6200-proyecto1-AI.git
%cd 2026-02-IC6200-proyecto1-AI/ia

Cloning into '2026-02-IC6200-proyecto1-AI'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 18 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 5.28 KiB | 1.76 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/2026-02-IC6200-proyecto1-AI/ia


In [ ]:
!cp /content/drive/MyDrive/Proyecto_IA/dataset.zip /content/
!unzip -q /content/dataset.zip -d /content/dataset_local/
!rm /content/dataset.zip

In [ ]:
import os
import torch
import torchaudio
from torchaudio.transforms import MelSpectrogram
from tqdm import tqdm

comandos = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']

# autodetectar la ruta correcta
ruta_base = '/content/dataset_local/'
if os.path.exists(os.path.join(ruta_base, 'archive')):
    ruta_base = os.path.join(ruta_base, 'archive/')

ruta_destino = '/content/drive/MyDrive/Proyecto_IA/tensores/'

os.makedirs(ruta_destino, exist_ok=True)

sample_rate = 16000
mel_spec = MelSpectrogram(sample_rate=sample_rate, n_mels=80, n_fft=400, hop_length=160)

x_data = []
y_label = []

# mapeo de texto a int
label_to_idx = {c: i for i, c in enumerate(comandos)}

for c in comandos:
    ruta_c = os.path.join(ruta_base, c)
    if not os.path.exists(ruta_c):
        continue

    archivos = [f for f in os.listdir(ruta_c) if f.endswith('.wav')]

    for archivo in tqdm(archivos, desc=f"Procesando {c}"):
        ruta_wav = os.path.join(ruta_c, archivo)

        wave, sr = torchaudio.load(ruta_wav)

        if sr != sample_rate:
            wave = torchaudio.transforms.Resample(orig_freq=sr, new_freq=sample_rate)(wave)

        # el modelo ocupa 1 segundo exacto de audio (16000 muestras)
        length = wave.shape[1]
        if length < sample_rate:
            wave = torch.nn.functional.pad(wave, (0, sample_rate - length))
        elif length > sample_rate:
            wave = wave[:, :sample_rate]

        spec = mel_spec(wave)

        x_data.append(spec)
        y_label.append(label_to_idx[c])

# apilar listas en tensores
X = torch.stack(x_data)
Y = torch.tensor(y_label)

torch.save((X, Y), os.path.join(ruta_destino, 'dataset_preprocesado.pt'))
print("Tensores guardados en Drive")

Procesando go: 100%|██████████| 3880/3880 [00:25<00:00, 152.43it/s]


Tensores guardados en Drive


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet5Audio(nn.Module):
    def __init__(self):
        super(LeNet5Audio, self).__init__()

        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # calculo del flatten para espectrogramas de 80x101:
        # conv1 -> pool1 -> conv2 -> pool2 deja el tensor en 16x17x22
        self.fc1 = nn.Linear(16 * 17 * 22, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # 10 clases de salida

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))

        x = x.view(-1, 16 * 17 * 22)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

# test rapido del shape
if __name__ == '__main__':
    modelo = LeNet5Audio()
    test_tensor = torch.randn(1, 1, 80, 101)
    print("Shape salida:", modelo(test_tensor).shape)

Shape salida: torch.Size([1, 10])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# bloque de convolucion separable (basado en literatura de MobileNet)
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        # depthwise: aplica un filtro por cada canal de entrada
        self.depthwise = nn.Conv2d(in_c, in_c, kernel_size=3, stride=stride, padding=1, groups=in_c)
        # pointwise: combina los canales usando un kernel 1x1
        self.pointwise = nn.Conv2d(in_c, out_c, kernel_size=1)
        self.bn = nn.BatchNorm2d(out_c)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = F.relu(self.bn(x))
        return x

class MobileNetAudio(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.features = nn.Sequential(
            DepthwiseSeparableConv(16, 32, stride=2),
            DepthwiseSeparableConv(32, 64, stride=2),
            DepthwiseSeparableConv(64, 128, stride=2)
        )

        # pooling adaptativo (siempre saca 1x1 sin importar el tamaño que llegue)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 10) # 10 logits de salida

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.features(x)
        x = self.pool(x)
        x = x.view(-1, 128)
        x = self.fc(x)
        return x

if __name__ == '__main__':
    modelo_b = MobileNetAudio()
    test_tensor = torch.randn(1, 1, 80, 101)
    print("Shape salida Modelo B:", modelo_b(test_tensor).shape)

Shape salida Modelo B: torch.Size([1, 10])


In [ ]:
!pip install wandb -q

import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchaudio.transforms as T
import copy

wandb.login()

ruta_tensores = '/content/drive/MyDrive/Proyecto_IA/tensores/dataset_preprocesado.pt'
x_full, y_full = torch.load(ruta_tensores)

class ComandosDataset(Dataset):
    def __init__(self, x, y, aplicar_aumento=False):
        self.x = x
        self.y = y
        self.aplicar_aumento = aplicar_aumento

        # mascaras para specaugment
        self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
        self.time_mask = T.TimeMasking(time_mask_param=35)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        spec = self.x[idx]

        if self.aplicar_aumento:
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)

        return spec, self.y[idx]

# particion 70/15/15
total = len(x_full)
train_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

dataset_full = ComandosDataset(x_full, y_full, aplicar_aumento=False)
train_dataset, val_dataset, test_dataset = random_split(
    dataset_full, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

def entrenar_modelo(nombre, tipo_red, aumentado, lr, batch_size, epochs):
    print(f"\n--- {nombre} | Aumentado: {aumentado} | LR: {lr} ---")

    train_dataset.dataset.aplicar_aumento = aumentado

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    modelo = tipo_red().to(device)

    criterio = nn.CrossEntropyLoss()
    optimizador = optim.Adam(modelo.parameters(), lr=lr)

    wandb.init(
        project="IC6200-Proyecto1",
        name=nombre,
        config={"lr": lr, "batch_size": batch_size, "epochs": epochs, "aumentado": aumentado}
    )

    mejor_val_loss = float('inf')
    mejor_modelo_wts = copy.deepcopy(modelo.state_dict())

    for epoch in range(epochs):
        modelo.train()
        train_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizador.zero_grad()
            outputs = modelo(inputs)
            loss = criterio(outputs, labels)
            loss.backward()
            optimizador.step()

            train_loss += loss.item() * inputs.size(0)

        train_loss = train_loss / len(train_loader.dataset)

        modelo.eval()
        val_loss = 0.0
        correctos = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = modelo(inputs)
                loss = criterio(outputs, labels)
                val_loss += loss.item() * inputs.size(0)

                _, preds = torch.max(outputs, 1)
                correctos += torch.sum(preds == labels.data)

        val_loss = val_loss / len(val_loader.dataset)
        val_acc = correctos.double() / len(val_loader.dataset)

        wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_acc": val_acc})

        print(f"Epoch {epoch}/{epochs-1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_loss < mejor_val_loss:
            mejor_val_loss = val_loss
            mejor_modelo_wts = copy.deepcopy(modelo.state_dict())

    wandb.finish()

    ruta_guardado = f"/content/drive/MyDrive/Proyecto_IA/modelos/{nombre}.pth"
    torch.save(mejor_modelo_wts, ruta_guardado)
    print(f"Guardado en {ruta_guardado}")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: danielzeasbrown (danielzeasbrown-instituto-tecnologico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# Modelo A (LeNet5) | Datos: Base | LR: 0.001 | Batch: 64 | Epochs: 15
entrenar_modelo("ModeloA_LeNet5_Base_Exp1", LeNet5Audio, False, 0.001, 64, 15)


--- ModeloA_LeNet5_Base_Exp1 | Aumentado: False | LR: 0.001 ---


Epoch 0/14 | Train Loss: 1.7368 | Val Loss: 1.1169 | Val Acc: 0.6276
Epoch 1/14 | Train Loss: 0.8511 | Val Loss: 0.7866 | Val Acc: 0.7473
Epoch 2/14 | Train Loss: 0.5475 | Val Loss: 0.6689 | Val Acc: 0.7789
Epoch 3/14 | Train Loss: 0.4220 | Val Loss: 0.6473 | Val Acc: 0.8083
Epoch 4/14 | Train Loss: 0.3489 | Val Loss: 0.6424 | Val Acc: 0.8346
Epoch 5/14 | Train Loss: 0.2600 | Val Loss: 0.6240 | Val Acc: 0.8421
Epoch 6/14 | Train Loss: 0.2237 | Val Loss: 0.6957 | Val Acc: 0.8377
Epoch 7/14 | Train Loss: 0.2135 | Val Loss: 0.8570 | Val Acc: 0.8279
Epoch 8/14 | Train Loss: 0.1606 | Val Loss: 0.7772 | Val Acc: 0.8412
Epoch 9/14 | Train Loss: 0.1569 | Val Loss: 0.7727 | Val Acc: 0.8384
Epoch 10/14 | Train Loss: 0.1134 | Val Loss: 0.8637 | Val Acc: 0.8414
Epoch 11/14 | Train Loss: 0.1242 | Val Loss: 0.8210 | Val Acc: 0.8348
Epoch 12/14 | Train Loss: 0.0900 | Val Loss: 0.8458 | Val Acc: 0.8530
Epoch 13/14 | Train Loss: 0.0913 | Val Loss: 0.9155 | Val Acc: 0.8493
Epoch 14/14 | Train Loss: 0.11

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_acc,▁▅▆▇▇██▇███▇███
val_loss,█▃▂▁▁▁▂▄▃▃▄▄▄▅▃
epoch,14
train_loss,0.11021
val_acc,0.8502
val_loss,0.77733


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloA_LeNet5_Base_Exp1.pth


In [ ]:
# Modelo A (LeNet5) | Datos: Base | LR: 0.0005 | Batch: 128 | Epochs: 15
entrenar_modelo("ModeloA_LeNet5_Base_Exp2", LeNet5Audio, False, 0.0005, 128, 15)


--- ModeloA_LeNet5_Base_Exp2 | Aumentado: False | LR: 0.0005 ---


Epoch 0/14 | Train Loss: 1.9782 | Val Loss: 1.5798 | Val Acc: 0.5115
Epoch 1/14 | Train Loss: 1.2431 | Val Loss: 1.1038 | Val Acc: 0.6430
Epoch 2/14 | Train Loss: 0.8275 | Val Loss: 0.9272 | Val Acc: 0.7189
Epoch 3/14 | Train Loss: 0.6201 | Val Loss: 0.8508 | Val Acc: 0.7677
Epoch 4/14 | Train Loss: 0.5054 | Val Loss: 0.8583 | Val Acc: 0.7743
Epoch 5/14 | Train Loss: 0.4016 | Val Loss: 0.8864 | Val Acc: 0.7859
Epoch 6/14 | Train Loss: 0.3593 | Val Loss: 0.9743 | Val Acc: 0.7822
Epoch 7/14 | Train Loss: 0.3179 | Val Loss: 0.9882 | Val Acc: 0.7999
Epoch 8/14 | Train Loss: 0.2979 | Val Loss: 1.0027 | Val Acc: 0.7983
Epoch 9/14 | Train Loss: 0.2578 | Val Loss: 1.0067 | Val Acc: 0.8095
Epoch 10/14 | Train Loss: 0.2115 | Val Loss: 1.0578 | Val Acc: 0.8094
Epoch 11/14 | Train Loss: 0.1787 | Val Loss: 1.1623 | Val Acc: 0.7987
Epoch 12/14 | Train Loss: 0.2054 | Val Loss: 1.1798 | Val Acc: 0.8007
Epoch 13/14 | Train Loss: 0.1711 | Val Loss: 1.1702 | Val Acc: 0.8028
Epoch 14/14 | Train Loss: 0.14

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▅▄▃▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▄▆▇▇▇▇████████
val_loss,█▃▂▁▁▁▂▂▂▂▃▄▄▄▅
epoch,14
train_loss,0.14352
val_acc,0.81716
val_loss,1.23119


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloA_LeNet5_Base_Exp2.pth


In [ ]:
# Modelo A (LeNet5) | Datos: Base | LR: 0.001 | Batch: 32 | Epochs: 10
entrenar_modelo("ModeloA_LeNet5_Base_Exp3", LeNet5Audio, False, 0.001, 32, 10)


--- ModeloA_LeNet5_Base_Exp3 | Aumentado: False | LR: 0.001 ---


wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch 0/9 | Train Loss: 1.5737 | Val Loss: 0.9851 | Val Acc: 0.6691
Epoch 1/9 | Train Loss: 0.7747 | Val Loss: 0.7600 | Val Acc: 0.7544
Epoch 2/9 | Train Loss: 0.5283 | Val Loss: 0.6268 | Val Acc: 0.7945
Epoch 3/9 | Train Loss: 0.4056 | Val Loss: 0.6286 | Val Acc: 0.8194
Epoch 4/9 | Train Loss: 0.3297 | Val Loss: 0.6917 | Val Acc: 0.8087
Epoch 5/9 | Train Loss: 0.2695 | Val Loss: 0.6901 | Val Acc: 0.8291
Epoch 6/9 | Train Loss: 0.2209 | Val Loss: 0.7932 | Val Acc: 0.8220
Epoch 7/9 | Train Loss: 0.1860 | Val Loss: 0.7863 | Val Acc: 0.8256
Epoch 8/9 | Train Loss: 0.1681 | Val Loss: 1.0130 | Val Acc: 0.8179
Epoch 9/9 | Train Loss: 0.1391 | Val Loss: 1.0047 | Val Acc: 0.8300


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▂▂▂▁▁▁▁
val_acc,▁▅▆█▇███▇█
val_loss,▇▃▁▁▂▂▄▄██
epoch,9
train_loss,0.13909
val_acc,0.82996
val_loss,1.00469


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloA_LeNet5_Base_Exp3.pth


In [ ]:
# Modelo A (LeNet5) | Datos: Aumentado | LR: 0.001 | Batch: 64 | Epochs: 15
entrenar_modelo("ModeloA_LeNet5_Aumentado_Exp1", LeNet5Audio, True, 0.001, 64, 15)


--- ModeloA_LeNet5_Aumentado_Exp1 | Aumentado: True | LR: 0.001 ---


Epoch 0/14 | Train Loss: 1.9305 | Val Loss: 1.4538 | Val Acc: 0.5098
Epoch 1/14 | Train Loss: 1.2962 | Val Loss: 1.1659 | Val Acc: 0.6108
Epoch 2/14 | Train Loss: 1.0343 | Val Loss: 1.0836 | Val Acc: 0.6388
Epoch 3/14 | Train Loss: 0.9131 | Val Loss: 0.9711 | Val Acc: 0.6817
Epoch 4/14 | Train Loss: 0.8110 | Val Loss: 0.9363 | Val Acc: 0.7033
Epoch 5/14 | Train Loss: 0.7438 | Val Loss: 0.9798 | Val Acc: 0.6912
Epoch 6/14 | Train Loss: 0.6974 | Val Loss: 0.8494 | Val Acc: 0.7248
Epoch 7/14 | Train Loss: 0.6577 | Val Loss: 0.8450 | Val Acc: 0.7218
Epoch 8/14 | Train Loss: 0.6314 | Val Loss: 0.8273 | Val Acc: 0.7263
Epoch 9/14 | Train Loss: 0.6055 | Val Loss: 0.8555 | Val Acc: 0.7346
Epoch 10/14 | Train Loss: 0.5662 | Val Loss: 0.8024 | Val Acc: 0.7449
Epoch 11/14 | Train Loss: 0.5533 | Val Loss: 0.7538 | Val Acc: 0.7521
Epoch 12/14 | Train Loss: 0.5363 | Val Loss: 0.7757 | Val Acc: 0.7513
Epoch 13/14 | Train Loss: 0.4978 | Val Loss: 0.7765 | Val Acc: 0.7558
Epoch 14/14 | Train Loss: 0.49

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁
val_acc,▁▄▅▆▆▆▇▇▇▇█████
val_loss,█▅▄▃▃▃▂▂▂▂▂▁▁▁▁
epoch,14
train_loss,0.49932
val_acc,0.75869
val_loss,0.74917


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloA_LeNet5_Aumentado_Exp1.pth


In [ ]:
# Modelo A (LeNet5) | Datos: Aumentado | LR: 0.0005 | Batch: 128 | Epochs: 15
entrenar_modelo("ModeloA_LeNet5_Aumentado_Exp2", LeNet5Audio, True, 0.0005, 128, 15)


--- ModeloA_LeNet5_Aumentado_Exp2 | Aumentado: True | LR: 0.0005 ---


Epoch 0/14 | Train Loss: 2.1235 | Val Loss: 1.8943 | Val Acc: 0.3747
Epoch 1/14 | Train Loss: 1.6604 | Val Loss: 1.5103 | Val Acc: 0.4997
Epoch 2/14 | Train Loss: 1.3126 | Val Loss: 1.2690 | Val Acc: 0.5810
Epoch 3/14 | Train Loss: 1.1117 | Val Loss: 1.1516 | Val Acc: 0.6201
Epoch 4/14 | Train Loss: 1.0041 | Val Loss: 1.1182 | Val Acc: 0.6482
Epoch 5/14 | Train Loss: 0.9198 | Val Loss: 1.0469 | Val Acc: 0.6663
Epoch 6/14 | Train Loss: 0.8564 | Val Loss: 1.0517 | Val Acc: 0.6615
Epoch 7/14 | Train Loss: 0.8125 | Val Loss: 1.0013 | Val Acc: 0.6812
Epoch 8/14 | Train Loss: 0.7595 | Val Loss: 0.9338 | Val Acc: 0.7058
Epoch 9/14 | Train Loss: 0.7429 | Val Loss: 0.9807 | Val Acc: 0.6855
Epoch 10/14 | Train Loss: 0.7029 | Val Loss: 0.9027 | Val Acc: 0.7030
Epoch 11/14 | Train Loss: 0.6683 | Val Loss: 0.9417 | Val Acc: 0.7054
Epoch 12/14 | Train Loss: 0.6447 | Val Loss: 0.9193 | Val Acc: 0.7218
Epoch 13/14 | Train Loss: 0.6234 | Val Loss: 0.8938 | Val Acc: 0.7191
Epoch 14/14 | Train Loss: 0.59

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▆▄▃▃▂▂▂▂▂▁▁▁▁▁
val_acc,▁▄▅▆▇▇▇▇█▇█████
val_loss,█▅▄▃▃▂▂▂▁▂▁▁▁▁▁
epoch,14
train_loss,0.59527
val_acc,0.72185
val_loss,0.91485


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloA_LeNet5_Aumentado_Exp2.pth


In [ ]:
# Modelo A (LeNet5) | Datos: Aumentado | LR: 0.001 | Batch: 32 | Epochs: 10
entrenar_modelo("ModeloA_LeNet5_Aumentado_Exp3", LeNet5Audio, True, 0.001, 32, 10)


--- ModeloA_LeNet5_Aumentado_Exp3 | Aumentado: True | LR: 0.001 ---


Epoch 0/9 | Train Loss: 1.8492 | Val Loss: 1.3518 | Val Acc: 0.5558
Epoch 1/9 | Train Loss: 1.1805 | Val Loss: 1.0911 | Val Acc: 0.6163
Epoch 2/9 | Train Loss: 0.9409 | Val Loss: 0.9293 | Val Acc: 0.6926
Epoch 3/9 | Train Loss: 0.8288 | Val Loss: 0.8915 | Val Acc: 0.7052
Epoch 4/9 | Train Loss: 0.7420 | Val Loss: 0.8084 | Val Acc: 0.7298
Epoch 5/9 | Train Loss: 0.6904 | Val Loss: 0.8602 | Val Acc: 0.7225
Epoch 6/9 | Train Loss: 0.6416 | Val Loss: 0.8204 | Val Acc: 0.7302
Epoch 7/9 | Train Loss: 0.6095 | Val Loss: 0.7908 | Val Acc: 0.7404
Epoch 8/9 | Train Loss: 0.5834 | Val Loss: 0.7593 | Val Acc: 0.7594
Epoch 9/9 | Train Loss: 0.5559 | Val Loss: 0.7301 | Val Acc: 0.7651


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▂▂▂▁▁▁▁
val_acc,▁▃▆▆▇▇▇▇██
val_loss,█▅▃▃▂▂▂▂▁▁
epoch,9
train_loss,0.5559
val_acc,0.76509
val_loss,0.73007


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloA_LeNet5_Aumentado_Exp3.pth


In [ ]:
# Modelo B (MobileNet) | Datos: Base | LR: 0.001 | Batch: 64 | Epochs: 15
entrenar_modelo("ModeloB_MobileNet_Base_Exp1", MobileNetAudio, False, 0.001, 64, 15)


--- ModeloB_MobileNet_Base_Exp1 | Aumentado: False | LR: 0.001 ---


Epoch 0/14 | Train Loss: 2.2153 | Val Loss: 2.1401 | Val Acc: 0.2122
Epoch 1/14 | Train Loss: 1.8046 | Val Loss: 4.5605 | Val Acc: 0.1744
Epoch 2/14 | Train Loss: 1.4262 | Val Loss: 3.4494 | Val Acc: 0.2923
Epoch 3/14 | Train Loss: 1.1639 | Val Loss: 2.2980 | Val Acc: 0.3724
Epoch 4/14 | Train Loss: 1.0089 | Val Loss: 3.6745 | Val Acc: 0.2334
Epoch 5/14 | Train Loss: 0.9077 | Val Loss: 3.0188 | Val Acc: 0.2508
Epoch 6/14 | Train Loss: 0.8319 | Val Loss: 1.9674 | Val Acc: 0.4546
Epoch 7/14 | Train Loss: 0.7698 | Val Loss: 2.6387 | Val Acc: 0.3311
Epoch 8/14 | Train Loss: 0.7203 | Val Loss: 3.5772 | Val Acc: 0.2494
Epoch 9/14 | Train Loss: 0.6810 | Val Loss: 2.6498 | Val Acc: 0.3629
Epoch 10/14 | Train Loss: 0.6431 | Val Loss: 4.3183 | Val Acc: 0.1821
Epoch 11/14 | Train Loss: 0.6135 | Val Loss: 2.9246 | Val Acc: 0.3003
Epoch 12/14 | Train Loss: 0.5840 | Val Loss: 2.4776 | Val Acc: 0.2967
Epoch 13/14 | Train Loss: 0.5634 | Val Loss: 1.7250 | Val Acc: 0.4309
Epoch 14/14 | Train Loss: 0.53

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▆▅▄▃▃▂▂▂▂▁▁▁▁▁
val_acc,▂▁▄▆▂▃█▅▃▆▁▄▄▇▆
val_loss,▂█▅▂▆▄▂▃▆▃▇▄▃▁▂
epoch,14
train_loss,0.53613
val_acc,0.38696
val_loss,2.23311


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloB_MobileNet_Base_Exp1.pth


In [ ]:
# Modelo B (MobileNet) | Datos: Base | LR: 0.0005 | Batch: 128 | Epochs: 15
entrenar_modelo("ModeloB_MobileNet_Base_Exp2", MobileNetAudio, False, 0.0005, 128, 15)


--- ModeloB_MobileNet_Base_Exp2 | Aumentado: False | LR: 0.0005 ---


Epoch 0/14 | Train Loss: 2.2734 | Val Loss: 2.2506 | Val Acc: 0.1977
Epoch 1/14 | Train Loss: 2.1420 | Val Loss: 2.0370 | Val Acc: 0.3003
Epoch 2/14 | Train Loss: 1.8725 | Val Loss: 1.9224 | Val Acc: 0.2827
Epoch 3/14 | Train Loss: 1.6828 | Val Loss: 2.0005 | Val Acc: 0.3025
Epoch 4/14 | Train Loss: 1.5214 | Val Loss: 1.8602 | Val Acc: 0.3543
Epoch 5/14 | Train Loss: 1.3746 | Val Loss: 1.8485 | Val Acc: 0.3745
Epoch 6/14 | Train Loss: 1.2548 | Val Loss: 1.3164 | Val Acc: 0.5473
Epoch 7/14 | Train Loss: 1.1526 | Val Loss: 2.6303 | Val Acc: 0.3323
Epoch 8/14 | Train Loss: 1.0756 | Val Loss: 1.5155 | Val Acc: 0.4949
Epoch 9/14 | Train Loss: 1.0027 | Val Loss: 2.7662 | Val Acc: 0.2795
Epoch 10/14 | Train Loss: 0.9471 | Val Loss: 1.9893 | Val Acc: 0.3899
Epoch 11/14 | Train Loss: 0.8983 | Val Loss: 3.1157 | Val Acc: 0.2776
Epoch 12/14 | Train Loss: 0.8520 | Val Loss: 1.7435 | Val Acc: 0.4456
Epoch 13/14 | Train Loss: 0.8189 | Val Loss: 2.5648 | Val Acc: 0.3318
Epoch 14/14 | Train Loss: 0.78

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▇▆▅▄▄▃▃▂▂▂▂▁▁▁
val_acc,▂▃▃▄▄▅█▄▇▃▅▃▆▄▁
val_loss,▄▃▃▃▃▃▁▅▂▆▃▇▂▅█
epoch,14
train_loss,0.7845
val_acc,0.16468
val_loss,3.46922


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloB_MobileNet_Base_Exp2.pth


In [ ]:
# Modelo B (MobileNet) | Datos: Base | LR: 0.001 | Batch: 32 | Epochs: 10
entrenar_modelo("ModeloB_MobileNet_Base_Exp3", MobileNetAudio, False, 0.001, 32, 10)


--- ModeloB_MobileNet_Base_Exp3 | Aumentado: False | LR: 0.001 ---


Epoch 0/9 | Train Loss: 1.9861 | Val Loss: 2.7355 | Val Acc: 0.1597
Epoch 1/9 | Train Loss: 1.3534 | Val Loss: 4.4793 | Val Acc: 0.1278
Epoch 2/9 | Train Loss: 1.0560 | Val Loss: 2.5089 | Val Acc: 0.2702
Epoch 3/9 | Train Loss: 0.8962 | Val Loss: 2.1786 | Val Acc: 0.3569
Epoch 4/9 | Train Loss: 0.7896 | Val Loss: 2.3450 | Val Acc: 0.2932
Epoch 5/9 | Train Loss: 0.7256 | Val Loss: 3.2354 | Val Acc: 0.2823
Epoch 6/9 | Train Loss: 0.6722 | Val Loss: 3.5312 | Val Acc: 0.2612
Epoch 7/9 | Train Loss: 0.6248 | Val Loss: 1.8106 | Val Acc: 0.4567
Epoch 8/9 | Train Loss: 0.5904 | Val Loss: 3.0547 | Val Acc: 0.2212
Epoch 9/9 | Train Loss: 0.5662 | Val Loss: 1.7263 | Val Acc: 0.4300


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▃▃▂▂▂▁▁▁
val_acc,▂▁▄▆▅▄▄█▃▇
val_loss,▄█▃▂▃▅▆▁▄▁
epoch,9
train_loss,0.56617
val_acc,0.43003
val_loss,1.72629


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloB_MobileNet_Base_Exp3.pth


In [ ]:
# Modelo B (MobileNet) | Datos: Aumentado | LR: 0.001 | Batch: 64 | Epochs: 15
entrenar_modelo("ModeloB_MobileNet_Aumentado_Exp1", MobileNetAudio, True, 0.001, 64, 15)


--- ModeloB_MobileNet_Aumentado_Exp1 | Aumentado: True | LR: 0.001 ---


Epoch 0/14 | Train Loss: 2.2340 | Val Loss: 2.2179 | Val Acc: 0.1514
Epoch 1/14 | Train Loss: 1.9710 | Val Loss: 5.6187 | Val Acc: 0.1197
Epoch 2/14 | Train Loss: 1.7427 | Val Loss: 4.3009 | Val Acc: 0.1560
Epoch 3/14 | Train Loss: 1.5781 | Val Loss: 6.9504 | Val Acc: 0.1067
Epoch 4/14 | Train Loss: 1.4622 | Val Loss: 6.9372 | Val Acc: 0.1500
Epoch 5/14 | Train Loss: 1.3846 | Val Loss: 4.3222 | Val Acc: 0.2190
Epoch 6/14 | Train Loss: 1.3112 | Val Loss: 3.9900 | Val Acc: 0.1892
Epoch 7/14 | Train Loss: 1.2649 | Val Loss: 5.2130 | Val Acc: 0.2482
Epoch 8/14 | Train Loss: 1.2138 | Val Loss: 5.7871 | Val Acc: 0.1503
Epoch 9/14 | Train Loss: 1.1714 | Val Loss: 11.0246 | Val Acc: 0.1451
Epoch 10/14 | Train Loss: 1.1469 | Val Loss: 4.8012 | Val Acc: 0.1958
Epoch 11/14 | Train Loss: 1.1138 | Val Loss: 2.7328 | Val Acc: 0.2904
Epoch 12/14 | Train Loss: 1.0879 | Val Loss: 4.1437 | Val Acc: 0.2446
Epoch 13/14 | Train Loss: 1.0541 | Val Loss: 4.5341 | Val Acc: 0.2148
Epoch 14/14 | Train Loss: 1.0

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▆▅▄▃▃▃▂▂▂▂▁▁▁▁
val_acc,▃▁▃▁▃▅▄▆▃▂▄█▆▅▆
val_loss,▁▄▃▅▅▃▂▃▄█▃▁▃▃▃
epoch,14
train_loss,1.04136
val_acc,0.24286
val_loss,4.23838


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloB_MobileNet_Aumentado_Exp1.pth


In [ ]:
# Modelo B (MobileNet) | Datos: Aumentado | LR: 0.0005 | Batch: 128 | Epochs: 15
entrenar_modelo("ModeloB_MobileNet_Aumentado_Exp2", MobileNetAudio, True, 0.0005, 128, 15)


--- ModeloB_MobileNet_Aumentado_Exp2 | Aumentado: True | LR: 0.0005 ---


Epoch 0/14 | Train Loss: 2.2893 | Val Loss: 2.2816 | Val Acc: 0.1548
Epoch 1/14 | Train Loss: 2.2299 | Val Loss: 2.1961 | Val Acc: 0.1969
Epoch 2/14 | Train Loss: 2.1081 | Val Loss: 2.0607 | Val Acc: 0.2830
Epoch 3/14 | Train Loss: 1.9939 | Val Loss: 3.0334 | Val Acc: 0.1399
Epoch 4/14 | Train Loss: 1.8969 | Val Loss: 5.1243 | Val Acc: 0.1188
Epoch 5/14 | Train Loss: 1.8202 | Val Loss: 3.2684 | Val Acc: 0.1711
Epoch 6/14 | Train Loss: 1.7429 | Val Loss: 6.9411 | Val Acc: 0.1607
Epoch 7/14 | Train Loss: 1.6789 | Val Loss: 5.5512 | Val Acc: 0.1673
Epoch 8/14 | Train Loss: 1.6175 | Val Loss: 5.6037 | Val Acc: 0.1806
Epoch 9/14 | Train Loss: 1.5653 | Val Loss: 5.9757 | Val Acc: 0.1771
Epoch 10/14 | Train Loss: 1.5158 | Val Loss: 5.9020 | Val Acc: 0.2058
Epoch 11/14 | Train Loss: 1.4760 | Val Loss: 5.1947 | Val Acc: 0.2109
Epoch 12/14 | Train Loss: 1.4352 | Val Loss: 5.7291 | Val Acc: 0.1437
Epoch 13/14 | Train Loss: 1.3981 | Val Loss: 5.6058 | Val Acc: 0.1301
Epoch 14/14 | Train Loss: 1.36

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,██▇▆▅▄▄▃▃▃▂▂▂▁▁
val_acc,▃▄█▂▁▃▃▃▄▃▅▅▂▁▂
val_loss,▁▁▁▂▅▃█▆▆▇▇▅▆▆▆
epoch,14
train_loss,1.36281
val_acc,0.14738
val_loss,5.30835


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloB_MobileNet_Aumentado_Exp2.pth


In [ ]:
# Modelo B (MobileNet) | Datos: Aumentado | LR: 0.001 | Batch: 32 | Epochs: 10
entrenar_modelo("ModeloB_MobileNet_Aumentado_Exp3", MobileNetAudio, True, 0.001, 32, 10)


--- ModeloB_MobileNet_Aumentado_Exp3 | Aumentado: True | LR: 0.001 ---


Epoch 0/9 | Train Loss: 2.1610 | Val Loss: 2.7396 | Val Acc: 0.1495
Epoch 1/9 | Train Loss: 1.8562 | Val Loss: 5.6440 | Val Acc: 0.1233
Epoch 2/9 | Train Loss: 1.6405 | Val Loss: 2.7508 | Val Acc: 0.2458
Epoch 3/9 | Train Loss: 1.4924 | Val Loss: 2.8330 | Val Acc: 0.2074
Epoch 4/9 | Train Loss: 1.3864 | Val Loss: 5.0471 | Val Acc: 0.1545
Epoch 5/9 | Train Loss: 1.3114 | Val Loss: 3.4350 | Val Acc: 0.1809
Epoch 6/9 | Train Loss: 1.2475 | Val Loss: 2.6513 | Val Acc: 0.2946
Epoch 7/9 | Train Loss: 1.1933 | Val Loss: 3.3880 | Val Acc: 0.2491
Epoch 8/9 | Train Loss: 1.1479 | Val Loss: 3.3620 | Val Acc: 0.2524
Epoch 9/9 | Train Loss: 1.1084 | Val Loss: 3.9532 | Val Acc: 0.2121


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▅▄▃▂▂▂▁▁
val_acc,▂▁▆▄▂▃█▆▆▅
val_loss,▁█▁▁▇▃▁▃▃▄
epoch,9
train_loss,1.10836
val_acc,0.21207
val_loss,3.9532


Guardado en /content/drive/MyDrive/Proyecto_IA/modelos/ModeloB_MobileNet_Aumentado_Exp3.pth


In [14]:
# instalar dependencias de onnx requeridas por colab
!pip install onnx onnxscript -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchaudio.transforms import MelSpectrogram

# definicion del modelo base original
class LeNet5Audio(nn.Module):
    def __init__(self):
        super(LeNet5Audio, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # calculo del flatten para espectrogramas de 80x101:
        self.fc1 = nn.Linear(16 * 17 * 22, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 17 * 22)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# wrapper para calcular el espectrograma internamente (contrato onnx)
class SmartHomeVoiceModel(nn.Module):
    def __init__(self, modelo_base):
        super().__init__()
        self.mel_transform = MelSpectrogram(
            sample_rate=16000,
            n_mels=80,
            n_fft=400,
            hop_length=160
        )
        self.modelo_base = modelo_base

    def forward(self, x):
        # asegurar dimension [1, 1, 16000] para el audio crudo
        x = x.view(1, 1, -1)
        spec = self.mel_transform(x)
        logits = self.modelo_base(spec)
        return logits

# cargar pesos del mejor modelo
ruta_modelo = '/content/drive/MyDrive/Proyecto_IA/modelos/ModeloA_LeNet5_Base_Exp3.pth'
mejor_modelo = LeNet5Audio()
mejor_modelo.load_state_dict(torch.load(ruta_modelo, weights_only=True))

modelo_wrapper = SmartHomeVoiceModel(mejor_modelo)
modelo_wrapper.eval()

# exportar usando opset 17 y un tensor dummy de 1 seg
ruta_onnx = '/content/drive/MyDrive/Proyecto_IA/smart_home_voice.onnx'
dummy_input = torch.randn(16000, dtype=torch.float32)

torch.onnx.export(
    modelo_wrapper,
    dummy_input,
    ruta_onnx,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=['audio_in'],
    output_names=['logits_out']
)

print(f"Exportado exitosamente a: {ruta_onnx}")

[torch.onnx] Obtain model graph for `SmartHomeVoiceModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SmartHomeVoiceModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exportado exitosamente a: /content/drive/MyDrive/Proyecto_IA/smart_home_voice.onnx
